# Nhận diện Mũ Bảo Hộ (Hard Hat Detection) với YOLO11

## 1. Kiểm tra môi trường và phần cứng (GPU/CPU)

In [1]:
import torch
import ultralytics
from ultralytics import YOLO

ultralytics.checks()
print(f'CUDA available: {torch.cuda.is_available()}')

ModuleNotFoundError: No module named 'ultralytics'

## 2. Kiểm tra cấu trúc dữ liệu và tệp `data.yaml`

In [ ]:
from pathlib import Path
import yaml

data_yaml_path = Path('datasets/hard-hat/data.yaml')
assert data_yaml_path.exists(), f'Không tìm thấy {data_yaml_path}. Hãy kiểm tra lại thư mục datasets!'

with open(data_yaml_path, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

print('Cấu hình dataset hiện tại:', data_cfg)

## 3. Huấn luyện mô hình (Train)

In [ ]:
import shutil

model = YOLO('yolo11s.pt')
results = model.train(
    data=str(data_yaml_path),
    epochs=50,
    imgsz=640,
    batch=8,
    project='runs/detect',
    name='train_exp',
    exist_ok=True
)

# Lưu trọng số tốt nhất vào thư mục models/
Path('models').mkdir(exist_ok=True)
best_pt = Path('runs/detect/train_exp/weights/best.pt')
if best_pt.exists():
    shutil.copy(best_pt, 'models/best.pt')
    print('Đã lưu mô hình tốt nhất tại: models/best.pt')

## 4. Đánh giá mô hình (Evaluate trên tập Test)

In [ ]:
best_model = YOLO('models/best.pt')

metrics = best_model.val(
    data=str(data_yaml_path),
    split='test',
    imgsz=640,
    conf=0.25,
    plots=True,
    project='runs/detect',
    name='val_exp',
    exist_ok=True
)

print('\n--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---')
print(f'mAP@50:      {metrics.box.map50:.4f}')
print(f'mAP@50-95:   {metrics.box.map:.4f}')
print(f'Precision:   {metrics.box.mp:.4f}')
print(f'Recall:      {metrics.box.mr:.4f}')

## 5. Trực quan hóa biểu đồ huấn luyện & Ma trận nhầm lẫn

In [ ]:
from IPython.display import Image, display

results_img = Path('runs/detect/train_exp/results.png')
cm_img = Path('runs/detect/val_exp/confusion_matrix.png')

if results_img.exists():
    display(Image(filename=str(results_img), width=800))
if cm_img.exists():
    display(Image(filename=str(cm_img), width=650))

## 6. Chạy nhận diện thử nghiệm (Predict) và hiển thị kết quả

In [ ]:
import glob

best_model.predict(
    source='datasets/hard-hat/test/images',
    conf=0.5,
    save=True,
    project='runs/detect',
    name='predict_exp',
    exist_ok=True
)

pred_images = sorted(glob.glob('runs/detect/predict_exp/*.jpg'))[:3]
for img_path in pred_images:
    display(Image(filename=img_path, width=600))